# BlackCode FR 260M - Entraînement dual-T4 Kaggle (6Md) — ULTRA OPTIMISÉ v2

Pipeline optimisé max v2 :
- Dataset 4 Go FineWeb2-HQ fra_Latn avec filtre français heuristique (×50) + qualité + no-near + tokenizer 20M chars (5 min)
- Tokenizer 24k français + 88milSMS CC BY 4.0 (optionnel, abréviations)
- Train 6 Md tokens en dual-T4 DDP avec Muon + WSD + selective checkpoint + compile max-autotune + DDP no_sync + fused + RMSNorm fused + cudnn benchmark + TF32 + prefetch 2 + bucket 10 MB

Temps estimé v2 : 45 min build + 12h train 6Md = ~16h total (vs 19h v1, vs 30h+ baseline) — dans quota 30h

## Gains v2 vs v1
- DDP no_sync grad-accum : +15-30% dual-T4
- Fused AdamW + RMSNorm fused F.rms_norm : +5-10% chaque
- compile max-autotune + cudnn benchmark + TF32 : +10-25% +5-10% +10-20%
- Prefetch 2 + bucket_cap 10 MB : +5-15% +5%
- TokenDataset torch.randint : +10%
- Tokenizer 20M chars + TOKENIZERS_PARALLELISM : 5 min vs 30 min


In [ ]:
!pip install datasets tokenizers -q
# lingua optionnel, fallback heuristique ×50 plus rapide
!pip install lingua-language-detector -q || echo "lingua non dispo, fallback heuristique activé"
!git clone https://github.com/Snowoo-2z/Llm-search.git /tmp/Llm-search || true
%cd /tmp/Llm-search
!ls configs/blackcode_fr_260m.json


In [ ]:
# Audit rapide 500 lignes (heuristique 0.1 ms vs lingua 5 ms → ×50)
!python scripts/audit_fr_source.py --dataset epfml/FineWeb2-HQ:fra_Latn --limit 500 --sample-count 10


In [ ]:
# Build ultra-optimisé v2 4 Go — 45 min vs 1h30 v1
# Optimisations : no-near (×2 RAM÷10), tokenizer 20M chars (5 min vs 30 min), TOKENIZERS_PARALLELISM=true, tar vers Drive
!MAX_MB=4000 MIN_QUALITY_SCORE=0.6 OUT=data/fr_260m HF_DATASET=epfml/FineWeb2-HQ:fra_Latn VOCAB=24576 SEQ_LEN=2048 TOKENIZER_CHARS=20000000 bash scripts/build_fr_optimized_v2.sh


In [ ]:
# Optionnel : ajoute 88milSMS CC BY 4.0 pour abréviations RS (slt/bjr/mdr)
!INCLUDE_SMS=1 MAX_MB=4000 bash scripts/build_fr_optimized_v2.sh


In [ ]:
# Vérif pureté + benchmark qualité (3435 docs/s)
!python scripts/eval_french_purity.py --corpus-dir data/fr_260m --limit 50
!python scripts/benchmark_quality.py
!cat data/fr_260m/meta.json | python3 -m json.tool | head -40


In [ ]:
# Train pilote 200 steps pour mesurer MFU avec optimisations v2
!python -m blackcode.train --config configs/blackcode_fr_260m.json --data data/fr_260m --out runs/fr_260m_pilot --batch-size 2 --grad-accum 8 --seq-len 2048 --max-steps 200 --optim adafactor --lr 3e-4 --fused --checkpoint-policy selective --compile --compile-mode max-autotune --cudnn-benchmark --tf32 --prefetch 2 --eval-interval 100 --save-interval 100 --resume --log-flops


In [ ]:
# Train complet 6 Md en DDP dual-T4 ULTRA OPTIMISÉ v2 (Kaggle)
# 6Md / (2*8*2048) = 183105 steps
# Gains v2 : no_sync + fused + rms_norm fused + max-autotune + cudnn benchmark + TF32 + prefetch + bucket 10
# Estimation : 12h vs 15h v1 vs 30h+ baseline
!torchrun --nproc_per_node=2 -m blackcode.train --config configs/blackcode_fr_260m.json --data data/fr_260m --out runs/fr_260m_6b --batch-size 2 --grad-accum 8 --seq-len 2048 --max-steps 183105 --optim muon --muon-lr 0.02 --lr 6e-4 --fused --schedule wsd --stable-frac 0.8 --seq-len-warmup-frac 0.2 --checkpoint-policy selective --compile --compile-mode max-autotune --cudnn-benchmark --tf32 --prefetch 2 --bucket-cap-mb 10 --ddp --log-flops --eval-interval 1000 --save-interval 1000 --mirror-dir /kaggle/working/drive/runs --resume


In [ ]:
# Alternative mono-T4 ultra optimisé (si Kaggle non dispo)
# 6Md / (2*16*2048) = 91552 steps avec grad-accum 16
!python -m blackcode.train --config configs/blackcode_fr_260m.json --data data/fr_260m --out runs/fr_260m_6b_mono --batch-size 2 --grad-accum 16 --seq-len 2048 --max-steps 91552 --optim muon --muon-lr 0.02 --lr 6e-4 --fused --schedule wsd --stable-frac 0.8 --seq-len-warmup-frac 0.2 --checkpoint-policy selective --compile --compile-mode max-autotune --cudnn-benchmark --tf32 --prefetch 2 --log-flops --eval-interval 1000 --save-interval 1000 --resume
